# D5 — Putting it together / Juntándolo todo

🇬🇧 Time to *prove* it. We train three models on the same long-range retrieval task and watch which ones beat
the over-squashing bottleneck:

| model | reach | weights | |
|---|---|---|---|
| **GAT** | 1-hop | learned | ordinary graph attention |
| **walkraw** | multi-hop (path algebra) | **fixed** = path count | reaches, but can't select |
| **WalkAttention** | multi-hop (path algebra) | **learned** | our model — reaches *and* selects |

🇪🇸 Es hora de *demostrarlo*. Entrenamos tres modelos en la misma tarea de recuperación de largo alcance y
miramos cuáles vencen el cuello de botella del over-squashing:

| modelo | alcance | pesos | |
|---|---|---|---|
| **GAT** | 1 salto | aprendidos | atención de grafos ordinaria |
| **walkraw** | multi-salto (álgebra) | **fijos** = conteo | alcanza, pero no selecciona |
| **WalkAttention** | multi-salto (álgebra) | **aprendidos** | nuestro modelo — alcanza *y* selecciona |

> 🇬🇧 This notebook runs a **small, fast** version (1 seed, fewer epochs) so you see the effect in ~2 minutes.
> The full 5-seed result (`WalkAttention = 1.000 ± 0.000`) is in `../05_walk_attention.ipynb`.
>
> 🇪🇸 Este cuaderno corre una versión **pequeña y rápida** (1 seed, menos épocas) para que veas el efecto en
> ~2 minutos. El resultado completo de 5 seeds (`WalkAttention = 1.000 ± 0.000`) está en
> `../05_walk_attention.ipynb`.

## The task / La tarea

🇬🇧 Each graph: `K` sources behind a bottleneck of depth `d`. One source holds the answer (its `key` matches a
query placed on the target). The target must output that source's identity. The deeper `d`, the worse the
squashing. Chance level is `1/K`.

🇪🇸 Cada grafo: `K` fuentes detrás de un cuello de botella de profundidad `d`. Una fuente tiene la respuesta
(su `key` coincide con una consulta puesta en el objetivo). El objetivo debe decir qué fuente es. A mayor `d`,
peor el apretujamiento. El nivel de azar es `1/K`.

In [ ]:
import torch, torch.nn.functional as F, numpy as np, pandas as pd, math
from torch.utils.data import DataLoader
from torch_geometric.loader import DataLoader as PyGLoader
from oversquash.data_bottleneck import make_bottleneck_graph
from oversquash.transforms import (AttachWalkMasks, collate_walk_masks,
                                   AttachWalkOperators, collate_walk_operators)
from oversquash.models import build_model

K, M, DEPTH = 5, 4, 3      # chance = 1/5 = 0.20
def ds(seed, n=1500):
    g = torch.Generator().manual_seed(seed)
    return [make_bottleneck_graph(K, M, DEPTH, g) for _ in range(n)]
print(f'task: {K} sources, depth {DEPTH}, chance = {1/K:.2f}')

In [ ]:
def train_eval(m, tr, va, fwd, epochs=80, lr=2e-3, warmup=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    def lr_lambda(e):
        if e < warmup: return (e+1)/warmup
        p=(e-warmup)/max(1,epochs-warmup); return 0.5*(1+math.cos(math.pi*p))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    def ev():
        m.eval(); a=[]
        with torch.no_grad():
            for b in va:
                lg,_=fwd(b); mm=b.root_mask
                a.append((lg[mm].argmax(-1)==b.y[mm]).float().mean().item())
        return float(np.mean(a))
    best=-1
    for e in range(epochs):
        m.train()
        for b in tr:
            opt.zero_grad(); lg,_=fwd(b)
            loss=F.cross_entropy(lg,b.y,ignore_index=-100); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
        sch.step(); best=max(best, ev())
    return best

def run(name, seed=0):
    torch.manual_seed(seed); data = ds(seed); nl = DEPTH+1
    if name=='gat':
        tr=PyGLoader(data[:1200],batch_size=128,shuffle=True); va=PyGLoader(data[1200:],batch_size=128)
        fwd=lambda b: m(b.x,b.edge_index,getattr(b,'batch',None))
    elif name=='walkraw':
        tf=AttachWalkOperators(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_operators)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_operators)
        fwd=lambda b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_raw=b.walk_raw)
    else:
        tf=AttachWalkMasks(n_layers=nl); data=[tf(d) for d in data]
        tr=DataLoader(data[:1200],batch_size=128,shuffle=True,collate_fn=collate_walk_masks)
        va=DataLoader(data[1200:],batch_size=128,collate_fn=collate_walk_masks)
        fwd=lambda b: m(b.x,b.edge_index,getattr(b,'batch',None),walk_masks=b.walk_masks)
    in_dim,out_dim=data[0].x.size(-1),data[0].num_classes
    m=build_model(name,in_dim,16,out_dim,nl,heads=4)
    return train_eval(m,tr,va,fwd)

In [ ]:
results = {name: run(name) for name in ['gat','walkraw','walkattn']}
for name, acc in results.items():
    print(f'{name:10s}: accuracy = {acc:.3f}')

In [ ]:
import matplotlib.pyplot as plt
names=['gat','walkraw','walkattn']; labels=['GAT\n(1-hop, learned)','walkraw\n(multi-hop, fixed)','WalkAttention\n(multi-hop, learned)']
vals=[results[n] for n in names]; cols=['#fca5a5','#fcd34d','#6ee7b7']
plt.figure(figsize=(6,3.6))
plt.bar(labels, vals, color=cols, edgecolor='#334155')
plt.axhline(1/K, ls='--', color='grey', label=f'chance = {1/K:.2f}')
for i,v in enumerate(vals): plt.text(i, v+0.02, f'{v:.2f}', ha='center', fontweight='bold')
plt.ylim(0,1.1); plt.ylabel('accuracy'); plt.legend()
plt.title(f'long-range retrieval, depth {DEPTH}'); plt.tight_layout(); plt.show()

## What you should see / Lo que deberías ver

🇬🇧 **GAT** near chance (1-hop attention can't cross the bottleneck). **walkraw** clearly above chance but
stuck (it reaches but can't select). **WalkAttention** highest — often perfect. The single ingredient that
separates `walkraw` from `WalkAttention` is *learned vs. fixed weights over the very same path-algebra
support*. That is the whole thesis.

🇪🇸 **GAT** cerca del azar (la atención de 1 salto no cruza el cuello de botella). **walkraw** claramente sobre
el azar pero estancado (alcanza pero no selecciona). **WalkAttention** el más alto — a menudo perfecto. El
único ingrediente que separa `walkraw` de `WalkAttention` es *pesos aprendidos vs. fijos sobre exactamente el
mismo soporte del álgebra de caminos*. Esa es toda la tesis.

---

🇬🇧 **The whole story, end to end:** over-squashing crams exponentially many messages into one vector (D1);
the path algebra counts those paths exactly (D2); each path is a message (D3); aggregating over walks has the
shape of attention, with the algebra supplying a sparse multi-hop *support* (D4); and learning the weights
over that support solves the task (D5). **The path algebra of a quiver defines a sparse multi-hop attention
that mitigates over-squashing.**

🇪🇸 **La historia completa, de principio a fin:** el over-squashing aprieta exponencialmente muchos mensajes
en un vector (D1); el álgebra de caminos cuenta esos caminos exactamente (D2); cada camino es un mensaje (D3);
agregar sobre recorridos tiene la forma de la atención, con el álgebra aportando un *soporte* sparse
multi-salto (D4); y aprender los pesos sobre ese soporte resuelve la tarea (D5). **El álgebra de caminos de un
quiver define una atención sparse multi-salto que mitiga el over-squashing.**